# 18A Appendix: Tokenizers, TF-IDF Variants, Word2Vec and Attention


[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/18A-Appendix-NLP-Details.ipynb)

This appendix holds the detail behind [Lecture 18](18-NLP.qmd): a short history of the field, the subword tokenizers that modern language models use (with the Hugging Face tokenizer demo), what scikit-learn's `TfidfVectorizer` actually computes and why its table differs from the textbook one, how Word2Vec learns a static embedding, and the transformer: its architecture, the attention mechanism with queries, keys and values, and the encoder-only, decoder-only and encoder–decoder families. The lecture *states* each of these; this is where they are spelled out. The quiz asks you to build a count vector, compute a TF-IDF weight and a cosine from given numbers, and explain the sparse-versus-dense idea, not to reproduce the formulas here.

Runnable examples of the NLP toolkits (NLTK, spaCy, BERTopic, LangChain) are in [Appendix A5: NLP Packages](A5-NLP-Packages.qmd).

# A Brief History of NLP

## NLP Evolution Timeline

**1950s-1970s: Rule-Based Era**

- Turing Test (1950), formal grammars, hand-crafted rules
- Systems like ELIZA (1966) - pattern matching "Rogerian therapy" chatbot
- Limitations: brittle, didn't scale, couldn't handle ambiguity

**1980s-2000s: Statistical Revolution**

- Shift from rules to learning from data
- Vector-space retrieval with TF-IDF, N-grams, Hidden Markov Models

**2006-2017: Deep Learning**

- Neural networks applied to NLP
- Word2Vec (2013), RNNs, LSTMs for sequence modeling
- Attention mechanism (2014) - precursor to Transformers

**2017-Present: Transformer Era**

- Attention is All You Need (2017) - Transformer architecture
- BERT (2018), GPT-2/3 (2019/2020), ChatGPT (2022)
- GPT-4 (2023), open-weight models (Llama, Mistral), "reasoning" models (2024–)


The first half of the lecture (bag of words, TF-IDF, cosine retrieval) lives in the statistical era; the second half (embeddings, language models, transformers) in the last two.

# Tokenizers in Practice

## Tokenization Methods

The lecture showed the two extremes: characters and words.

* Characters give a tiny vocabulary, but a single character carries almost no meaning, so the model has to reassemble words itself. For English (a non-character-based language) we certainly lose meaning.
* Words carry meaning, but the vocabulary is enormous and any word not seen in training is simply *unknown*.

In between are **subword** methods, which is what modern language models use:

 - [Byte-Pair Encoding](https://en.wikipedia.org/wiki/Byte-pair_encoding),
 - [SentencePiece](https://github.com/google/sentencepiece).

See Andrej Karpathy's [Let's build a GPT tokenizer](https://www.youtube.com/watch?v=zduSFxRajkE) video for a deep dive. With tokenization, the goal is to not lose meaning with the tokens.

## Hugging Face Tokenization

Here is a demo of how to tokenize using the [transformers](https://huggingface.co/docs/transformers/en/index) package from Hugging Face. This is BERT's own tokenizer.

In [ ]:
#| echo: false
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()

In [ ]:
#| code-fold: false
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")

sentence = "Show me the money"
tokens = tokenizer.tokenize(sentence)
print(tokens)

# Try a more advanced sentence
sentence2 = "Let's try to see if we can get this transformer to tokenize."
tokens2 = tokenizer.tokenize(sentence2)
print(tokens2)
print("vocabulary size:", tokenizer.vocab_size)

`##` means "continues the previous token": `token` + `##ize` = *tokenize*. BERT's
vocabulary has about 29,000 **subwords**, and any word can be spelled from them, so the
model never meets an unknown word. That is why subword tokenizers won: a fixed, manageable vocabulary, and no unknown words.

## Tokens, Token IDs, and Vocabulary

* Associated to each token is a unique token ID.

* The total number of unique tokens that a model can recognize and process is the *vocabulary size*.

  * The *vocabulary* is the collection of all the unique tokens.

* The tokens (and token ids) alone hold no (semantic) information. Token 2,048 is not "more" than token 17; it is an arbitrary index. What is needed is a numerical representation that *encodes* this information.

* There are different ways to achieve this:

  - One encoding technique that we already considered is one-hot encodings.

  - Another more powerful encoding method, is the creation of word embeddings.

## tiktokenizer

![tiktokenizer](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/figs/tiktokenizer.png)

The [Tiktokenizer demo](https://tiktokenizer.vercel.app/) lets you paste text and see how different models split it into tokens, each token coloured. Try your own name, a long technical word, or a sentence in another language: common English words are one token each, and everything else gets chopped up.

# What `TfidfVectorizer` Actually Computes

## The three sentences again

The lecture's three sentences, as a bag of words and as textbook TF-IDF (so this appendix runs on its own):

In [ ]:
#| code-fold: false
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

docs = ["The cat sat on the mat.",
        "The dog sat on the log.",
        "The emu sat on the mat."]

count_vec = CountVectorizer()              # lower-cases, drops punctuation
counts = count_vec.fit_transform(docs).toarray()
vocab = count_vec.get_feature_names_out()

N = len(docs)
df = (counts > 0).sum(axis=0)
idf = np.log(N / df)
tfidf_textbook = counts * idf
pd.DataFrame(tfidf_textbook, columns=vocab, index=["S1", "S2", "S3"]).round(3)

## scikit-learn's formula

scikit-learn does **not** use the textbook formula. Its defaults are

$$
\mathrm{idf}_{\text{sk}}(t) = \ln\frac{1+N}{1+\mathrm{df}(t)} + 1,
\qquad\text{then each row is divided by its } \ell_2 \text{ norm.}
$$

In [ ]:
#| code-fold: false
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec = TfidfVectorizer()
tfidf_sk = tfidf_vec.fit_transform(docs).toarray()
pd.DataFrame(tfidf_sk, columns=tfidf_vec.get_feature_names_out(),
             index=["S1", "S2", "S3"]).round(4)

## Why `the` is the largest entry

- **Smoothing** (the $+1$s): `the` gets $\mathrm{idf}_{\text{sk}} = \ln(4/4) + 1 = 1$, not 0.
  With tf $= 2$ its raw weight is $2$, which outweighs `cat` at $\ln 2 + 1 = 1.69$ times a count of 1. No term is ever weighted exactly 0,
  and a word that is new at query time cannot cause a division by zero.
- **$\ell_2$ normalization**: every row has length 1, so long and short documents are
  comparable and cosine similarity becomes a dot product. That is why `the` reads $0.6165$ in S1 and S3 but $0.5839$ in S2: the rows have different raw lengths before normalization.
- `smooth_idf=False` removes the $+1$s inside the log but keeps the $+1$ outside, so it is
  still not the textbook idf.
- Practical knobs: `stop_words="english"`, `min_df` / `max_df` to prune the vocabulary,
  `sublinear_tf=True` for $1 + \ln \mathrm{tf}$.
- On thousands of documents the smoothing barely matters. On three it does.

## The same numbers, checked by hand

The smoothed idf for each term, and the unnormalized and normalized weights of sentence 1:

In [ ]:
#| code-fold: false
idf_sk = np.log((1 + N) / (1 + df)) + 1
raw_s1 = counts[0] * idf_sk
pd.DataFrame({"idf_sk": idf_sk.round(4),
              "S1 raw": raw_s1.round(4),
              "S1 normalized": (raw_s1 / np.linalg.norm(raw_s1)).round(4)},
             index=vocab).T

The normalized row is the first row of the `TfidfVectorizer` table above.

# Word2Vec in Detail

## Word2Vec: Static Embeddings

**Word2Vec** [@mikolov2013efficient] is a technique to learn static word embeddings from large text corpora.

**Key characteristics:**

- Each word has a **single fixed vector** representation
- Trained to predict a word from its context (**CBOW**) or the context from a word (**Skip-gram**): a shallow network whose hidden weights become the vectors, so words that keep the same company end up close together
- Captures semantic relationships: *king - man + woman ≈ queen*. The offset from "man" to "woman" is roughly the same vector as from "king" to "queen": semantic relations are encoded as directions, not just nearest neighbours
- Popular pre-trained model: Google News (300d). Related: GloVe, which is count-based
  (it factorizes global word co-occurrence counts) rather than predictive

**Advantages:**

- Fast to train and use
- Efficient for large vocabularies
- Good for similarity tasks, clustering
- Pre-trained embeddings available

**Disadvantages:**

- Limited context and no word order information.
- For phrases and sentences, the embedding is the average of the word embeddings.


## When to use which


**When to use Word2Vec:**

- Limited computational resources
- Task doesn't require deep context understanding
- Working with domain-specific corpora (train custom embeddings)


**When to use Contextual Embeddings:**

- Need to handle polysemy (words with multiple meanings)
- Complex NLP tasks (NER, question answering, translation)
- Have access to GPU resources


**The bridge you already know: LSA.** Take the SVD of the TF-IDF matrix ([Lecture 12](12-SVD-Low-Rank.qmd)) and keep the top $k$ directions ([Lecture 13](13-Dimensionality-Reduction-PCA-tSNE.qmd)): each direction is a combination of terms that co-occur across documents, and projecting a document onto them gives a dense $k$-vector. "laptop" and "notebook" load on the same direction, so two tickets that share no word can still score above 0. That is the simplest, linear version of an embedding.

# Language Models

## N-gram and neural language models

A language model is a statistical tool that predicts the probability of a sequence of words. It helps in understanding and generating human language by learning patterns and structures from large text corpora.

1. **N-gram Models**:
   - Predict the next word based on the previous  $n-1$ words, by counting.
   - Simple and effective for many tasks but limited by fixed context size.

1. **Neural Language Models**:
   - Use neural networks to capture more complex patterns.
   - Examples include *RNNs*, *LSTMs*, and **Transformers**.

See [RNNs and LSTMs](M3-RNN.qmd) for more details. The rest of this appendix is about the transformer.

# Transformers in Detail

## Transformers: High-Level Overview

Transformers [@vaswani2017attention] are the foundation of modern NLP systems.

**Key innovations:**

- Use **attention mechanism** to process entire sequences (e.g. context window) in parallel
  - Attention costs $O(n^2)$ in the sequence length $n$; [Flash Attention](https://arxiv.org/abs/2205.14135) (chunking) and sparse attention make long contexts affordable.
- Revolutionized NLP and enabled LLMs (BERT, GPT-4, ChatGPT)
- Scalable across GPUs for massive models

## Transformer Architecture

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/drawio/Transformer_Enc_Dec.png)

**Components:**

- **Encoder**: Processes input text, creates rich representations
- **Decoder**: Generates output text using encoder representations
- **Attention**: Allows model to focus on relevant parts of input

> Other variants of the transformer architecture include the encoder-only and decoder-only architectures.

## Encoder-Decoder Building Blocks

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/drawio/TransformerOriginal.png)

> Can grow the model size by increasing the number of layers and the number of attention heads.

## Transformer Building Block


**Components:**

- Attention mechanism -- gathers information from neighboring tokens in the input sequence.
- Feed-forward neural network -- processes the information gathered by the attention mechanism.
- Layer normalization -- normalizes the output of each sub-layer (attention and feed-forward).

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/drawio/Encoder.png)


## The Attention Mechanism

Attention allows models to understand which words are most relevant to each other.

**Example sentence:**

> "The elephant didn't cross the river because **it** was tired."

**Question:** What does "it" refer to?

- Attention helps the model determine that **"it" = "elephant"** (not "river")
- The model "attends" more strongly to "elephant" when processing "it"

## How Attention Works (Simplified)

For each word in a sentence:

1. **Calculate relevance scores** with all other words
2. **Apply [softmax](https://en.wikipedia.org/wiki/Softmax_function)** to get attention weights (sum to 1)
3. **Compute weighted sum** of word representations
4. Result: each word's representation includes context from relevant words

**Multi-head attention**: Run multiple attention operations in parallel to capture different types of relationships

> It's a form of adaptive network where some weights are derived from inputs.

## Queries, Keys, and Values

The attention mechanism uses three types of vectors for each word:

- **Query (Q)**: "What am I looking for?"
- **Key (K)**: "What do I contain?"
- **Value (V)**: "What information do I have?"

**Attention formula:**

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

- Compute similarity between queries and keys
- Normalize with softmax
- Weight the values by attention scores

This is why "bank" gets a different vector next to "river" than next to "account": the neighbours it attends to are different, so the weighted sum of values is different.

## Attention Visualized

Darker connections show stronger attention.

When processing "it", the model attends most to "elephant".

This is how transformers handle long-range dependencies and ambiguity.

See also [BertViz](https://github.com/jessevig/bertviz) for a visual exploration of the attention mechanism.

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/drawio/VisualizedAttention.png)


# Transformer Architectures

## 3 Types of Transformer Models

1. **Encoder-Decoder** – Sequence-to-sequence tasks (e.g., machine translation)
   - Example: Original Transformer, T5

2. **Encoder-Only** – Transforms text embeddings into latent representations for understanding and classification tasks
   - Example: **BERT** (sentiment analysis, NER, classification)

3. **Decoder-Only** – Predicts next token used for text generation and completion
   - Example: **GPT** (ChatGPT, text generation, AI assistants)

## BERT: Encoder Model

**BERT** (Bidirectional Encoder Representations from Transformers) [@devlin2019bert]

**Key features:**

- Reads text **bidirectionally** (considers context from both directions)
- Pre-trained on masked language modeling (predict hidden words)
- 340M parameters (BERT-Large), 110M (BERT-Base)
- Fine-tuned for specific tasks with one additional layer

**Common uses:** Text classification, NER, question answering, sentiment analysis

## GPT: Decoder Model

**GPT** (Generative Pre-trained Transformer) [@brown2020language]

**Key features:**

- Reads text **left-to-right** (autoregressive)
- Predicts next token: $P(t_1, t_2, \ldots, t_N) = P(t_1)\prod_{n=2}^{N} P(t_n | t_1, \ldots t_{n-1})$
- GPT-3: 175 billion parameters
- Excellent at text generation

**Common uses:** Text completion, creative writing, chatbots, code generation

## Transformer Applications

Transformers enable powerful NLP applications:

- **Machine Translation**: Translate between languages
- **Text Summarization**: Generate concise summaries
- **Question Answering**: Answer questions from context
- **Named Entity Recognition**: Identify entities in text
- **Sentiment Analysis**: Determine sentiment/emotion
- **Text Generation**: Create human-like text

Encoders do the understanding tasks on this list, decoders do the generating ones, and encoder-decoders do the ones that turn one text into another.

## Bibliography
